In [ ]:
import numpy as np
import pandas as pd
# Assuming trueLigaF2223 are already defined

# Define the range for home win coefficients
home_win_coeffs = np.arange(0, 1.26, 0.25)

# Define the range for away win coefficients
away_win_coeffs = np.arange(0.75, 2.01, 0.25)

# Define the range for away win coefficients
margin_coeffs = np.arange(0.0, 2.01, 0.50)

# Define the range for date  coefficients
date_coeffs = np.arange(0, 1.01, 0.5)

print("Home Win Coefficients:", home_win_coeffs)
print("Away Win Coefficients:", away_win_coeffs)
print("Margin Coefficients:", margin_coeffs)
print("Date Coefficients:", date_coeffs)

# Define the Colley function, specifically for weighting mode 4 optimization
def Colley_weighted_optimized(league, home_win_coeff, away_win_coeff, margin_coeff, date_coeff):
    hTeam = league['home_team']
    aTeam = league['away_team']

    teams = sorted(set(hTeam).union(set(aTeam)))
    A = 2 * np.eye(len(teams))
    b = np.ones(len(teams))
    w = np.ones(len(league))
    teamIndex = {team: idx for idx, team in enumerate(teams)}

    # Weighting mode 4 logic (optimized)
    def margin(s):
        s = str(s).replace('–', '-')
        try:
            x, y = [int(t) for t in s.split('-')]
            return min(3, abs(x - y)**margin_coeff)
        except:
            return 1
    w_margin = league['score'].apply(margin).to_numpy()

    latest_week = league['gameweek'].max()
    w_date = (league['gameweek'] / latest_week).to_numpy() ** date_coeff

    # Combine weights (e.g., multiply them)
    w = w_margin * w_date

    for i in range(len(league)):
        row = league.iloc[i]
        home, away = row['home_team'], row['away_team']
        score_txt = str(row['score']).replace('–', '-')

        try:
            hScore, aScore = [int(t) for t in score_txt.split('-')]
        except:
            continue

        hi = teamIndex[home]; ai = teamIndex[away]

        # Colley matrix
        A[hi, ai] -= w[i]; A[ai, hi] -= w[i]
        A[hi, hi] += w[i]; A[ai, ai] += w[i]

        # Vector b updates
        if hScore > aScore:
            b[hi] += home_win_coeff * w[i]; 
            b[ai] -= home_win_coeff * w[i]
        elif aScore > hScore:
            b[ai] += away_win_coeff * w[i];
            b[hi] -= away_win_coeff * w[i]
        # ties: no change


    r = np.linalg.solve(A, b)
    team_ratings = {team: r[idx] for idx, team in enumerate(teams)}
    sorted_teams = sorted(team_ratings, key=team_ratings.get, reverse=True)
    return team_ratings, sorted_teams

# Define Constant
best_error = float('inf')
best_mpd = float('inf')
best_home_coeff = None
best_date_coeff = None
best_away_coeff = None
best_margin_coeff = None


history = []   # will store one row per try
